# Notebook 03 — Modelos Baseline

## Objetivo

Entrenar modelos clásicos de clasificación supervisada sobre representaciones BoW y TF-IDF.

**Experimento principal:** subconjunto político.  
**Experimento complementario:** dataset completo (solo baselines).  
**Sub-experimento:** ablación de marcadores de fuente (`reuters`, `ap`, `afp` → `[SOURCE]`) sobre el mejor modelo; la decisión condiciona los experimentos 2+.

> El modelo aprende patrones lingüísticos del dataset; no verifica hechos.

**Métrica principal:** F2-score de la clase fake (prioriza recall de fake sin ignorar precisión).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.paths import *
from nlp.plotting import setup_style, save_figure

setup_style()

FIGURE_PREFIX = '03_'

from sklearn.metrics import confusion_matrix, roc_curve, auc
import joblib

from nlp.io import load_splits
from nlp.modeling import (
    baseline_text_columns,
    decide_source_normalization,
    evaluate_best_configs_on_test,
    predict_proba_scores,
    run_baseline_grid,
    run_source_ablation,
    save_source_ablation_decision,
)
from nlp.plotting import plot_confusion_matrix


## 1. Carga de splits

In [2]:
BASELINE_COLS = baseline_text_columns()
pol_train, pol_val, pol_test = load_splits('politics', columns=BASELINE_COLS)
full_train, full_val, full_test = load_splits('full', columns=BASELINE_COLS)


## 2. Configuración de la grilla de experimentos

In [3]:

NGRAM_OPTS = [(1, 1), (1, 2)]
MAX_FEATURES_OPTS = [10000, 30000, 50000]
VECTORIZER_TYPES = ['bow', 'tfidf']


## 3. Entrenamiento y selección por validación (F2 fake)

In [4]:
grid_kwargs = dict(
    ngram_opts=NGRAM_OPTS,
    max_features_opts=MAX_FEATURES_OPTS,
    vectorizer_types=VECTORIZER_TYPES,
)
if DEV_MODE:
    grid_kwargs['max_combos'] = 20
    print('DEV_MODE activo: grilla politics limitada a max_combos=20')

# Experimento principal: subconjunto político (solo métricas de validación)
politics_results = run_baseline_grid(pol_train, pol_val, 'politics', **grid_kwargs)


politics:   0%|          | 0/72 [00:00<?, ?it/s]

In [ ]:
if DEV_MODE:
    print('DEV_MODE activo: se omite la grilla full_dataset')
    full_results = pd.DataFrame()
else:
    full_results = run_baseline_grid(
        full_train, full_val, 'full_dataset',
        ngram_opts=NGRAM_OPTS,
        max_features_opts=MAX_FEATURES_OPTS,
        vectorizer_types=VECTORIZER_TYPES,
    )

val_results = politics_results if full_results.empty else pd.concat(
    [politics_results, full_results], ignore_index=True,
)
print('Mejores 10 configuraciones según validación:')
print(val_results.sort_values('f2_fake', ascending=False).head(10).to_string(index=False))


## 3b. Comparación por campo de texto

**Hipótesis:** el cuerpo o el texto completo (título + cuerpo) discriminan mejor que el título solo.

Para cada campo (`title`, `body`, `full`) se toma la configuración con mayor F2 en validación (subconjunto político) y se comparan en un gráfico.

In [ ]:
FIELD_LABELS = {'title': 'Título', 'body': 'Cuerpo', 'full': 'Título + cuerpo'}
FIELD_ORDER = ['title', 'body', 'full']

best_by_field = (
    politics_results
    .loc[politics_results.groupby('text_field')['f2_fake'].idxmax()]
    .set_index('text_field')
    .loc[FIELD_ORDER]
    .reset_index()
    .assign(text_field_label=lambda df: df['text_field'].map(FIELD_LABELS))
)

print('Mejor F2 en validación por campo de texto (politics):')
print(
    best_by_field[
        ['text_field', 'f2_fake', 'recall_fake', 'model', 'vectorizer', 'stopwords']
    ].to_string(index=False)
)

# Gráfico principal: mejor F2 por campo (hipótesis título vs cuerpo vs completo)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=best_by_field,
    x='text_field_label',
    y='f2_fake',
    hue='text_field_label',
    order=[FIELD_LABELS[f] for f in FIELD_ORDER],
    palette=['#95a5a6', '#3498db', '#2ecc71'],
    legend=False,
    ax=ax,
)
ymin = best_by_field['f2_fake'].min() - 0.008
ymax = best_by_field['f2_fake'].max() + 0.012
ax.set_ylim(ymin, ymax)
for i, row in enumerate(best_by_field.itertuples()):
    ax.text(i, row.f2_fake + (ymax - ymin) * 0.02, f'{row.f2_fake:.3f}', ha='center')
ax.set_ylabel('F2 fake (validación)')
ax.set_xlabel('Campo de texto')
ax.set_title('Mejor baseline por campo — politics (val)')
save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}baseline_f2_by_text_field.png')
plt.show()

# Referencia: top configuraciones globales (coloreadas por campo)
top_n = politics_results.nlargest(9, 'f2_fake').copy()
top_n['label'] = (
    top_n['model'].str.replace('_', ' ')
    + ' · ' + top_n['vectorizer']
    + ' · ' + top_n['text_field'].map(FIELD_LABELS)
)
fig, ax = plt.subplots(figsize=(9, 5))
plot_top = top_n.sort_values('f2_fake', ascending=True)
sns.barplot(
    data=plot_top,
    x='f2_fake',
    y='label',
    hue='text_field',
    dodge=False,
    palette={'title': '#95a5a6', 'body': '#3498db', 'full': '#2ecc71'},
    ax=ax,
)
ax.set_xlim(ymin - 0.005, ymax + 0.01)
ax.set_xlabel('F2 fake (validación)')
ax.set_ylabel('')
ax.set_title('Top 9 configuraciones — politics (val)')
ax.legend(title='Campo', labels=[FIELD_LABELS[f] for f in FIELD_ORDER])
save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}baseline_top_models.png')
plt.show()

## 4. Mejor modelo (selección en val) y evaluación final en test

In [ ]:

# Selección final por validación; test solo para el mejor modelo de cada alcance
pol_best_test_df, pol_best_val, pipe_pol, text_col = evaluate_best_configs_on_test(
    politics_results, pol_train, pol_test, 'politics',
)
pol_best_test = pol_best_test_df.iloc[0]

test_rows = [pol_best_test]
if not full_results.empty:
    full_best_test_df, _, _, _ = evaluate_best_configs_on_test(
        full_results, full_train, full_test, 'full_dataset',
    )
    test_rows.append(full_best_test_df.iloc[0])

val_results = val_results.assign(source_condition=pd.NA)
baseline_results = pd.concat(
    [val_results, pd.DataFrame(test_rows)], ignore_index=True,
)

print('Mejor configuración politics (val):')
print(pol_best_val.to_string())
print('\nEvaluación en test del mejor modelo politics:')
print(pol_best_test.to_string())

y_pred = pipe_pol.predict(pol_test[text_col].fillna(''))
cm = confusion_matrix(pol_test['label'], y_pred)

plot_confusion_matrix(
    cm, ['Real', 'Fake'],
    f'Matriz de confusión — {pol_best_test["model"]}',
    RESULTS_FIGURES / f'{FIGURE_PREFIX}baseline_best_confusion_matrix.png',
)

X_test = pol_test[text_col].fillna('')
y_proba = predict_proba_scores(pipe_pol, X_test)

if y_proba is not None:
    fpr, tpr, _ = roc_curve(pol_test['label'], y_proba)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('Curva ROC — mejor baseline (politics, test)')
    ax.legend()
    save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}baseline_best_roc_curve.png')
    plt.show()

joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_politics.joblib')
pol_best_test.to_json(RESULTS_MODELS / 'best_baseline_politics_config.json')
joblib.dump(pipe_pol, RESULTS_MODELS / 'best_baseline_model.joblib')
print('Modelos guardados en results/models/')


## 5. Ablación de marcadores de fuente

Tras identificar el mejor modelo en validación, se reentrena con la misma configuración en dos condiciones:

- **A (original):** texto con menciones de agencias (`reuters`, `ap`, `afp`).
- **B (normalizado):** esos tokens reemplazados por `[SOURCE]`.

Si el F2 cae significativamente en B (Δ ≥ umbral en validación), los experimentos siguientes deben normalizar fuentes.

In [ ]:
ablation_results = run_source_ablation(
    pol_best_val, pol_train, pol_val, pol_test, dataset_scope='politics',
)

print('Ablación de fuente (mejor config politics):')
print(
    ablation_results[
        ['source_condition', 'split', 'f2_fake', 'recall_fake', 'precision_fake', 'roc_auc']
    ].to_string(index=False)
)

val_ablation = ablation_results[ablation_results['split'] == 'val'].set_index('source_condition')
f2_original = val_ablation.loc['original', 'f2_fake']
f2_normalized = val_ablation.loc['normalized', 'f2_fake']

decision = decide_source_normalization(f2_original, f2_normalized)
decision['best_config'] = pol_best_val.to_dict()
test_ablation = ablation_results[ablation_results['split'] == 'test'].set_index('source_condition')
decision['f2_original_test'] = float(test_ablation.loc['original', 'f2_fake'])
decision['f2_normalized_test'] = float(test_ablation.loc['normalized', 'f2_fake'])

save_source_ablation_decision(decision, SOURCE_ABLATION_DECISION)
ablation_results.to_csv(RESULTS_METRICS / 'source_ablation_results.csv', index=False)

# La ablación es un diagnóstico separado; baseline_results queda reservado
# para la grilla de validación y la evaluación final del mejor modelo.
baseline_results.to_csv(RESULTS_METRICS / 'baseline_results.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
plot_df = ablation_results[ablation_results['split'] == 'val']
sns.barplot(
    data=plot_df,
    x='source_condition',
    y='f2_fake',
    hue='source_condition',
    legend=False,
    ax=ax,
)
ax.set_ylabel('F2 fake (validación)')
ax.set_xlabel('Condición de texto')
ax.set_title('Ablación de marcadores de fuente — mejor baseline politics')
save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}baseline_source_ablation.png')
plt.show()

if decision['use_source_normalization']:
    print(
        f"Decisión: normalizar fuentes en experimentos 2+ "
        f"(caída F2 val = {decision['f2_drop']:.3f} ≥ {decision['threshold']})."
    )
else:
    print(
        f"Decisión: mantener texto original en experimentos 2+ "
        f"(caída F2 val = {decision['f2_drop']:.3f} < {decision['threshold']})."
    )
print(f'Decisión guardada en {SOURCE_ABLATION_DECISION}')

## 6. Comparación politics vs full_dataset

Si el rendimiento en el dataset completo es mucho mayor que en el subconjunto político, probablemente parte del rendimiento se explica por sesgos de tema, fuente o estructura del dataset — no solo por patrones lingüísticos de fake news.

In [ ]:

compare_val = val_results.groupby('dataset_scope')['f2_fake'].max()
compare_test = baseline_results[
    (baseline_results['split'] == 'test') & baseline_results['source_condition'].isna()
].set_index('dataset_scope')['f2_fake']
print('Mejor F2 en validación por alcance:')
print(compare_val)
print('\nF2 en test del mejor modelo seleccionado en val:')
print(compare_test)

fig, ax = plt.subplots(figsize=(8, 4))
top_pol = politics_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
if full_results.empty:
    sns.barplot(x=['politics (top-5 avg)'], y=[top_pol], ax=ax)
else:
    top_full = full_results.nlargest(5, 'f2_fake')['f2_fake'].mean()
    sns.barplot(
        x=['politics (top-5 avg)', 'full_dataset (top-5 avg)'],
        y=[top_pol, top_full],
        ax=ax,
    )
ax.set_ylabel('F2 fake en validación (promedio top-5)')
ax.set_title('Comparación subconjunto político vs dataset completo (val)')
save_figure(fig, RESULTS_FIGURES / f'{FIGURE_PREFIX}baseline_politics_vs_full.png')
plt.show()


## Conclusiones

- Se compararon LR, MNB y Linear SVM con BoW y TF-IDF.
- Se evaluaron título, cuerpo y título+cuerpo; con y sin stopwords.
- El gráfico por campo de texto contrasta la hipótesis de que cuerpo/completo superan al título solo (mejor F2 en val por `text_field`).
- La métrica principal F2 prioriza detectar fake news (minimizar falsos negativos).
- La ablación de fuente cuantifica si el rendimiento depende de marcadores de agencia; la decisión queda en `source_ablation_decision.json` para los experimentos 2+.
- El modelo aprende patrones del dataset, no verifica hechos.